# 5. cVAE + Adversarial Batch Correction (no KL)

**Architecture:** Encoder + batch-conditioned Decoder + GRL Discriminator.
`Loss = MSE_recon + CE(disc(GRL(z)), batch)`  — GRL scales gradient by -λ.

**Defaults:** lr=5e-4, λ_adv=0.5, warmup=40%, grad_clip=0.5, label_smoothing=0.1, 100 epochs

## 0. Setup

In [ ]:
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds
!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger
from matplotlib.lines import Line2D
from sklearn.metrics import silhouette_score
from umap import UMAP

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import CVAEAdvCorrector, CVAEAdvConfig

set_logger(level="INFO")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}", DEVICE)

## 1. Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_adv')
DATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')

SPLIT_COL = "Split_dummy"
ADV_KEY = SCVI_CORRECTED_KEY + "_adv"

MULTI_BATCH_COHORTS = ["PUB_KIRC_ICI_combined", "NSCLC_Tissue_ICI_Pred", "PUB_BRCA_SCANB"]
IMMOTION_ICI = "PUB_ccRCC_Immotion150_and_151_ICI"
IMMOTION_TKI = "PUB_ccRCC_Immotion150_and_151_TKI"
IMMOTION_COMBINED = "PUB_ccRCC_Immotion_combined"
BLCA_NAME = "PUB_BLCA_Mariathasan_EGAS00001002556"

## 2. Helpers

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.0)

def plot_umap_panel(coords, labels, title, ax, palette=None):
    unique = sorted(labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique))
    cmap = dict(zip(unique, palette))
    colors = [cmap[l] for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6,
               edgecolors="none", rasterized=True)
    ax.set_title(title, fontsize=10, fontweight="bold")
    if len(unique) <= 12:
        handles = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cmap[l], markersize=7, label=str(l))
                   for l in unique]
        ax.legend(handles=handles, fontsize=6, loc="best", framealpha=0.8)

def run_correction(adata, batch_col, config, adv_key):
    """Run adversarial correction: fit on train, transform all."""
    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)
    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch_labels = adata.obs[batch_col].astype(str).values

    logger.info("Batch distribution:\n{}", pd.Series(batch_labels).value_counts())
    logger.info("Split: train={}, test={}", train_mask.sum(), test_mask.sum())

    corrector = CVAEAdvCorrector(config=config)
    corrector.fit(X=emb_all[train_mask], batch_labels=batch_labels[train_mask])
    corrected = corrector.transform(emb_all)
    adata.obsm[adv_key] = corrected

    assert np.isfinite(corrected).all(), "NaN/Inf detected!"
    logger.info("Corrected: {}", corrected.shape)
    return adata, corrector

def normalize_obsm(adata):
    """Convert all obsm entries to numpy arrays (fixes concat issues)."""
    for key in list(adata.obsm.keys()):
        val = adata.obsm[key]
        if isinstance(val, pd.DataFrame):
            adata.obsm[key] = val.values
        elif not isinstance(val, np.ndarray):
            adata.obsm[key] = np.asarray(val)
    return adata

## 3. Process Multi-Batch Cohorts (KIRC, NSCLC)

In [ ]:
adv_config = CVAEAdvConfig(latent_dim=64, seed=SEED)
all_histories = {}

for cohort_name in MULTI_BATCH_COHORTS:
    src  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    dest = DATA_PROCESSED_ADV  / f"{cohort_name}.adata.zarr"
    if not src.exists():
        logger.warning("Missing: {}", cohort_name); continue
    if dest.exists():
        logger.info("Skip (exists): {}", cohort_name); continue

    logger.info("\n" + "=" * 60 + "\nProcessing: {}\n" + "=" * 60, cohort_name)
    adata = load_cohort(src)
    adata, corr = run_correction(adata, BATCH_COL, adv_config, ADV_KEY)
    all_histories[cohort_name] = corr.history_
    save_adata_zarr(adata, dest)
    logger.info("Saved: {}", dest)
    del adata
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("Multi-batch cohorts done.")

## 4. Immotion ICI + TKI Combined (Over-correction Control)

ICI and TKI are from the **same lab** (no real batch effect).
We combine them, treat ICI/TKI as 2 'batches', run correction.
**Expected:** silhouette by arm should NOT change (Δ ≈ 0).

In [ ]:
ici_path = DATA_INTERIM_PT_DIR / f"{IMMOTION_ICI}.adata.zarr"
tki_path = DATA_INTERIM_PT_DIR / f"{IMMOTION_TKI}.adata.zarr"
combined_dest = DATA_PROCESSED_ADV / f"{IMMOTION_COMBINED}.adata.zarr"

if ici_path.exists() and tki_path.exists() and not combined_dest.exists():
    ad_ici = load_cohort(ici_path)
    ad_tki = load_cohort(tki_path)

    # Mark which arm each sample is from (becomes our 'batch')
    ad_ici.obs["Immotion_arm"] = "ICI"
    ad_tki.obs["Immotion_arm"] = "TKI"

    # Normalize obsm to numpy arrays before concat (avoids DataFrame/array mismatch)
    ad_ici = normalize_obsm(ad_ici)
    ad_tki = normalize_obsm(ad_tki)

    # Concatenate
    adata_combined = ad.concat([ad_ici, ad_tki], merge="same")
    adata_combined.obs_names_make_unique()

    logger.info("Combined Immotion: {} samples", adata_combined.n_obs)
    logger.info("Arms: {}", adata_combined.obs["Immotion_arm"].value_counts().to_dict())

    # Run correction using Immotion_arm as batch label
    adata_combined, corr_imm = run_correction(
        adata_combined, "Immotion_arm", adv_config, ADV_KEY,
    )
    all_histories[IMMOTION_COMBINED] = corr_imm.history_

    # Silhouette by arm: before vs after
    emb_raw  = np.asarray(adata_combined.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_corr = np.asarray(adata_combined.obsm[ADV_KEY], dtype=np.float32)
    arm_labels = adata_combined.obs["Immotion_arm"].astype(str).values

    sil_raw  = silhouette_score(emb_raw, arm_labels)
    sil_corr = silhouette_score(emb_corr, arm_labels)
    print(f"\nImmotion combined over-correction check:")
    print(f"  Sil(ICI vs TKI) Before: {sil_raw:.4f}")
    print(f"  Sil(ICI vs TKI) After:  {sil_corr:.4f}")
    print(f"  Delta: {sil_corr - sil_raw:.4f}  (should be ~0)")

    diag = adata_combined.obs[DIAGNOSIS_COL].astype(str).values
    if len(set(diag)) > 1:
        sil_d_raw  = silhouette_score(emb_raw, diag)
        sil_d_corr = silhouette_score(emb_corr, diag)
        print(f"  Sil(Diagnosis) Before: {sil_d_raw:.4f}")
        print(f"  Sil(Diagnosis) After:  {sil_d_corr:.4f}")

    save_adata_zarr(adata_combined, combined_dest)
    logger.info("Saved: {}", combined_dest)
    del ad_ici, ad_tki, adata_combined
else:
    if combined_dest.exists():
        logger.info("Immotion combined already exists")
    else:
        logger.warning("Missing ICI or TKI")

## 5. BLCA Single-Batch (copy as-is)

In [ ]:
blca_src  = DATA_INTERIM_PT_DIR / f"{BLCA_NAME}.adata.zarr"
blca_dest = DATA_PROCESSED_ADV  / f"{BLCA_NAME}.adata.zarr"
if blca_src.exists() and not blca_dest.exists():
    adata = load_cohort(blca_src)
    emb = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    adata.obsm[ADV_KEY] = emb
    save_adata_zarr(adata, blca_dest)
    logger.info("BLCA: copied raw -> {}", blca_dest)
    del adata
else:
    logger.info("BLCA: done or missing")

## 6. Training Curves

In [ ]:
if not all_histories:
    logger.warning("No histories.")
else:
    for cname, hist in all_histories.items():
        fig, axes = plt.subplots(1, 4, figsize=(22, 4))
        fig.suptitle(f"{cname} -- Adv-cVAE", fontsize=13, fontweight="bold", y=1.03)
        ep = range(1, len(hist.total) + 1)
        axes[0].plot(ep, hist.recon, color='#2196F3', lw=1.5)
        axes[0].set_title("Recon (MSE)"); axes[0].set_xlabel("Epoch")
        axes[1].plot(ep, hist.adv, color='#FF5722', lw=1.5)
        axes[1].set_title("Adv (CE)"); axes[1].set_xlabel("Epoch")
        axes[2].plot(ep, hist.disc_accuracy, color='#9C27B0', lw=1.5)
        axes[2].set_title("Disc Acc"); axes[2].set_xlabel("Epoch"); axes[2].set_ylim(0,1)
        axes[3].plot(ep, hist.lambda_schedule, color='#4CAF50', lw=1.5)
        axes[3].set_title("Lambda"); axes[3].set_xlabel("Epoch")
        plt.tight_layout(); plt.show()

## 7. UMAP Before / After

In [ ]:
test_cohort = None
for name in MULTI_BATCH_COHORTS + [IMMOTION_COMBINED]:
    p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    if p.exists(): test_cohort = name; break

if test_cohort:
    adata = load_cohort(DATA_PROCESSED_ADV / f"{test_cohort}.adata.zarr")
    emb_raw = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    umap_raw = UMAP(n_components=2, random_state=SEED).fit_transform(emb_raw)
    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)
    umap_corr = UMAP(n_components=2, random_state=SEED).fit_transform(emb_corr)
    bcol = BATCH_COL if BATCH_COL in adata.obs and adata.obs[BATCH_COL].nunique() > 1 else "Immotion_arm"
    batch = adata.obs[bcol].astype(str)
    diag  = adata.obs[DIAGNOSIS_COL].astype(str)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{test_cohort} ({adata.n_obs} samples)", fontsize=14, fontweight="bold", y=1.02)
    plot_umap_panel(umap_raw,  batch, f"Before -- {bcol}", axes[0,0])
    plot_umap_panel(umap_corr, batch, f"After -- {bcol}",  axes[0,1])
    plot_umap_panel(umap_raw,  diag,  "Before -- Diagnosis", axes[1,0])
    plot_umap_panel(umap_corr, diag,  "After -- Diagnosis",  axes[1,1])
    for ax in axes.flat: ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    plt.tight_layout(); plt.show()
    del adata

## 8. Silhouette Scores

In [ ]:
rows = []
for name in MULTI_BATCH_COHORTS + [IMMOTION_COMBINED, BLCA_NAME]:
    p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    if not p.exists(): continue
    adata = load_cohort(p)
    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)
    if name == IMMOTION_COMBINED:
        bcol = "Immotion_arm"
    else:
        bcol = BATCH_COL
    batch = adata.obs[bcol].astype(str).values if bcol in adata.obs else None
    diag  = adata.obs[DIAGNOSIS_COL].astype(str).values
    mb = batch is not None and len(set(batch)) > 1
    md_ = len(set(diag)) > 1
    rows.append({
        "Cohort": name, "Samples": adata.n_obs,
        "Sil_Batch_Before": round(silhouette_score(emb_raw, batch), 4) if mb else np.nan,
        "Sil_Batch_After":  round(silhouette_score(emb_corr, batch), 4) if mb else np.nan,
        "Sil_Diag_Before":  round(silhouette_score(emb_raw, diag), 4) if md_ else np.nan,
        "Sil_Diag_After":   round(silhouette_score(emb_corr, diag), 4) if md_ else np.nan,
    })
    del adata
if rows:
    df_sil = pd.DataFrame(rows)
    df_sil["Batch_D"] = df_sil["Sil_Batch_After"] - df_sil["Sil_Batch_Before"]
    df_sil["Diag_D"]  = df_sil["Sil_Diag_After"]  - df_sil["Sil_Diag_Before"]
    display(df_sil)
    print("Batch_D < 0 = mixing (good) | Diag_D >= 0 = biology preserved (good)")

## 9. Mariathasan Permutation Test

BLCA: 1 real batch. Random labels from other cohorts' pool → train → Δ ≈ 0.

In [ ]:
blca_path = DATA_INTERIM_PT_DIR / f"{BLCA_NAME}.adata.zarr"
if blca_path.exists():
    adata = load_cohort(blca_path)
    emb = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    n = adata.n_obs
    diag = adata.obs[DIAGNOSIS_COL].astype(str).values
    has_diag = len(set(diag)) > 1

    batch_pool = []
    for cname in COHORT_CANCER_CODE:
        if cname == BLCA_NAME: continue
        p = DATA_INTERIM_PT_DIR / f"{cname}.adata.zarr"
        if not p.exists(): continue
        ad_tmp = load_cohort(p)
        batch_pool.extend(ad_tmp.obs[BATCH_COL].astype(str).values.tolist())
        del ad_tmp
    batch_pool = np.array(batch_pool)
    logger.info("Pool: {} samples, {} labels", len(batch_pool), len(set(batch_pool)))

    perm_results = []
    for i in range(5):
        rng = np.random.RandomState(SEED + i)
        fake = rng.choice(batch_pool, size=n, replace=True)
        if len(set(fake)) < 2: continue
        cfg = CVAEAdvConfig(latent_dim=64, seed=SEED + i)
        corr = CVAEAdvCorrector(config=cfg)
        corr.fit(X=emb, batch_labels=fake)
        emb_p = corr.transform(emb)
        row = {"perm": i, "n_fake": len(set(fake)),
               "sil_batch_before": round(silhouette_score(emb, fake), 4),
               "sil_batch_after":  round(silhouette_score(emb_p, fake), 4)}
        row["batch_D"] = round(row["sil_batch_after"] - row["sil_batch_before"], 4)
        if has_diag:
            row["sil_diag_before"] = round(silhouette_score(emb, diag), 4)
            row["sil_diag_after"]  = round(silhouette_score(emb_p, diag), 4)
            row["diag_D"] = round(row["sil_diag_after"] - row["sil_diag_before"], 4)
        perm_results.append(row)
    if perm_results:
        df_perm = pd.DataFrame(perm_results)
        display(df_perm)
        print(f"Mean batch_D: {df_perm['batch_D'].mean():.4f}")
        if 'diag_D' in df_perm:
            print(f"Mean diag_D:  {df_perm['diag_D'].mean():.4f} (should be ~0)")
    del adata

## 10. KL-cVAE vs Adversarial cVAE

In [ ]:
rows_cmp = []
for name in MULTI_BATCH_COHORTS:
    old_p = DATA_PROCESSED_DIR / f"{name}.adata.zarr"
    new_p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    if not old_p.exists() or not new_p.exists(): continue
    ad_old = load_cohort(old_p); ad_new = load_cohort(new_p)
    batch = ad_old.obs[BATCH_COL].astype(str).values
    diag  = ad_old.obs[DIAGNOSIS_COL].astype(str).values
    mb = len(set(batch)) > 1; md_ = len(set(diag)) > 1
    emb_raw = np.asarray(ad_old.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    r = {"Cohort": name,
         "Sil_Batch_Raw": round(silhouette_score(emb_raw, batch), 4) if mb else np.nan,
         "Sil_Diag_Raw":  round(silhouette_score(emb_raw, diag), 4) if md_ else np.nan}
    if SCVI_CORRECTED_KEY in ad_old.obsm:
        e = np.asarray(ad_old.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
        r["Sil_Batch_KL"] = round(silhouette_score(e, batch), 4) if mb else np.nan
        r["Sil_Diag_KL"]  = round(silhouette_score(e, diag), 4) if md_ else np.nan
    if ADV_KEY in ad_new.obsm:
        e = np.asarray(ad_new.obsm[ADV_KEY], dtype=np.float32)
        r["Sil_Batch_Adv"] = round(silhouette_score(e, batch), 4) if mb else np.nan
        r["Sil_Diag_Adv"]  = round(silhouette_score(e, diag), 4) if md_ else np.nan
    rows_cmp.append(r); del ad_old, ad_new
if rows_cmp:
    display(pd.DataFrame(rows_cmp))
    print("Ideal: Batch_Adv < Batch_KL AND Diag_Adv > Diag_KL")